<a href="https://colab.research.google.com/github/AD1t12407/Deeplernin/blob/main/LMV3_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install imgkit

In [2]:
!pip install pytorch-lightning

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [3]:
import os
import imgkit
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from torchmetrics import Accuracy
import json

# Install required libraries (ensure this is run in an environment where you have access to install packages)
!pip install imgkit
!apt-get install -y wkhtmltopdf

# Download and extract the dataset
!gdown 1tMZXonmajLPK9zhZ2dt-CdzRTs5YfHy0
!unzip -q financial-documents.zip
!mv "TableClassifierQuaterlyWithNotes" "documents"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  avahi-daemon bind9-host bind9-libs geoclue-2.0 glib-networking glib-networking-common
  glib-networking-services gsettings-desktop-schemas iio-sensor-proxy libavahi-core7 libavahi-glib1
  libdaemon0 libevdev2 libfontenc1 libgudev-1.0-0 libhyphen0 libinput-bin libinput10
  libjson-glib-1.0-0 libjson-glib-1.0-common liblmdb0 libmaxminddb0 libmbim-glib4 libmbim-proxy
  libmd4c0 libmm-glib0 libmtdev1 libnl-genl-3-200 libnotify4 libnss-mdns libproxy1v5 libqmi-glib5
  libqmi-proxy libqt5core5a libqt5dbus5 libqt5gui5 libqt5network5 libqt5positioning5
  libqt5printsupport5 libqt5qml5 libqt5qmlmodels5 libqt5quick5 libqt5sensors5 libqt5svg5
  libqt5webchannel5 libqt5webkit5 libqt5widgets5 libsoup2.4-1 libsoup2.4-common libudev1
  libwacom-bin libwacom-common libwacom9 libwoff1 libxcb-icccm4 libxcb-image0 libxcb-keysyms1
  libxcb-render-util0 libx

In [4]:
# Define the input and output directories
input_dir = "documents"
output_dir = "images"

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the subfolders (document classes)
document_classes = ['Balance Sheets', 'Cash Flow', 'Income Statement', 'Notes', 'Others']

# Traverse each subfolder and convert HTML files to images
for doc_class in document_classes:
    class_input_dir = os.path.join(input_dir, doc_class)
    class_output_dir = os.path.join(output_dir, doc_class)
    os.makedirs(class_output_dir, exist_ok=True)

    html_files = [f for f in os.listdir(class_input_dir) if f.endswith('.html')]
    for html_file in html_files:
        input_path = os.path.join(class_input_dir, html_file)
        output_path = os.path.join(class_output_dir, html_file.replace('.html', '.png'))
        imgkit.from_file(input_path, output_path)

print("Conversion completed!")

Streaming output truncated to the last 5000 lines.
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering 

In [5]:
!apt-get install tesseract-ocr
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 44 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 1s (4,717 kB/s)
Selecting previously unselected package tesseract-ocr-eng.
(Reading database ... 125863 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-

In [6]:
# Function to perform OCR and save results as JSON
import pytesseract # Import the pytesseract module
def create_json_from_image(image_path, json_path):
    image = Image.open(image_path).convert("RGB")
    ocr_data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)

    # Format the OCR data to match LayoutLMv3 expected format
    words = []
    boxes = []
    for i in range(len(ocr_data['level'])):
        if ocr_data['text'][i].strip():
            word = ocr_data['text'][i]
            box = [ocr_data['left'][i], ocr_data['top'][i], ocr_data['left'][i] + ocr_data['width'][i], ocr_data['top'][i] + ocr_data['height'][i]]
            words.append({"word": word, "bounding_box": box})

    with open(json_path, 'w') as json_file:
        json.dump(words, json_file)

In [ ]:

# Traverse each subfolder and convert HTML files to images
for doc_class in document_classes:
    class_input_dir = os.path.join(input_dir, doc_class)
    class_output_dir = os.path.join(output_dir, doc_class)
    os.makedirs(class_output_dir, exist_ok=True)

    # Get a list of all HTML files in the subdirectory
    html_files = [f for f in os.listdir(class_input_dir) if f.endswith('.html')]

    # Convert each HTML file to an image
    for html_file in html_files:
        input_path = os.path.join(class_input_dir, html_file)
        output_path = os.path.join(class_output_dir, html_file.replace('.html', '.png'))
        imgkit.from_file(input_path, output_path)

        # Create corresponding JSON file from the image
        json_path = os.path.join(class_output_dir, html_file.replace('.html', '.json'))
        create_json_from_image(output_path, json_path)

print("Conversion and JSON creation completed!")

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loading page (1/2)
Rendering (2/2)                                                    
Done                                                               
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-root'
Loadi

In [ ]:
data = []
for doc_class in document_classes[:2]:
    class_output_dir = os.path.join(output_dir, doc_class)
    images = [f for f in os.listdir(class_output_dir) if f.endswith('.png')]
    for image in images:
        json_path = os.path.join(class_output_dir, image.replace('.png', '.json'))
        if os.path.exists(json_path):
            data.append({
                'image_path': os.path.join(class_output_dir, image),
                'label': doc_class,
                'json_path': json_path
            })
        else:
            print(f"Warning: JSON file not found for {image}")

In [ ]:
# Create a DataFrame
df = pd.DataFrame(data)
print(df.head())  # Display the first few rows

# Split the data into training and testing sets
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
print(f'Training set size: {len(train_df)}')
print(f'Testing set size: {len(test_df)}')


In [ ]:
# Prepare processor
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base",apply_ocr=False)

# Map labels to IDs
label_to_id = {label: idx for idx, label in enumerate(sorted(df['label'].unique()))}

# Define Dataset Class
class DocumentClassificationDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.dataframe = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_path = self.dataframe.iloc[idx]['image_path']
        label = self.dataframe.iloc[idx]['label']
        json_path = self.dataframe.iloc[idx]['json_path']

        with open(json_path, 'r') as f:
            ocr_result = json.load(f)

        image = Image.open(image_path).convert("RGB")
        words = [item['word'] for item in ocr_result]
        boxes = [item['bounding_box'] for item in ocr_result]

        encoding = self.processor(
            image,
            words=words,
            boxes=boxes,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze() for k, v in encoding.items()}
        encoding['labels'] = torch.tensor(label_to_id[label], dtype=torch.long)

        return encoding

In [ ]:

# Define Dataset Class
class DocumentClassificationDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.dataframe = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        image_path = self.dataframe.iloc[idx]['image_path']
        label = self.dataframe.iloc[idx]['label']
        json_path = self.dataframe.iloc[idx]['json_path']

        with open(json_path, 'r') as f:
            ocr_result = json.load(f)

        image = Image.open(image_path).convert("RGB")

        # Ensure 'words' and 'boxes' are extracted correctly, handling potential missing keys
        words = [item.get('word', '') for item in ocr_result]  # Use .get() to handle missing 'word' keys
        boxes = [item.get('bounding_box', []) for item in ocr_result]  # Use .get() to handle missing 'bounding_box' keys

        encoding = self.processor(
            image,
            words=words,  # Pass the 'words' list to the processor
            boxes=boxes,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        encoding = {k: v.squeeze() for k, v in encoding.items()}
        encoding['labels'] = torch.tensor(label_to_id[label], dtype=torch.long)

        return encoding

In [ ]:
# Define Model
class DocumentClassificationModel(pl.LightningModule):
    def __init__(self, num_labels):
        super().__init__()
        self.model = LayoutLMv3ForSequenceClassification.from_pretrained(
            "microsoft/layoutlmv3-base", num_labels=num_labels
        )
        self.train_accuracy = Accuracy(task='multiclass', num_classes=num_labels)
        self.val_accuracy = Accuracy(task='multiclass', num_classes=num_labels)

    def forward(self, input_ids, attention_mask, bbox, pixel_values, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values,
            labels=labels
        )

    def training_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        self.log("train_loss", loss)
        self.log("train_accuracy", self.train_accuracy(outputs.logits, batch['labels']), prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(**batch)
        loss = outputs.loss
        self.log("val_loss", loss)
        self.log("val_accuracy", self.val_accuracy(outputs.logits, batch['labels']), prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-5)

In [ ]:
# Define DataLoaders
train_dataset = DocumentClassificationDataset(train_df, processor)
test_dataset = DocumentClassificationDataset(test_df, processor)

train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
test_data_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

# Initialize and train model
num_labels = len(label_to_id)
model = DocumentClassificationModel(num_labels)


In [ ]:
# Check if CUDA is available and set the device accordingly
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'
devices = 1 if torch.cuda.is_available() else 2

trainer = pl.Trainer(max_epochs=3, accelerator=accelerator, devices=devices)
trainer.fit(model, train_data_loader, test_data_loader)


In [ ]:
# Save the trained model
model_path = "contents/model"  # Replace with your desired save path
model.model.save_pretrained(model_path)
processor.save_pretrained(model_path)

print(f"Model and processor saved to {model_path}")


In [ ]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification
import torch
from PIL import Image
import json

# Define the path to the saved model and processor
model_path = "contents/model"

# Load the saved processor and model
processor = LayoutLMv3Processor.from_pretrained(model_path)
model = LayoutLMv3ForSequenceClassification.from_pretrained(model_path)

# Set the model to evaluation mode
model.eval()


In [ ]:
def prepare_test_data(image_path, json_path, processor):
    with open(json_path, 'r') as f:
        ocr_result = json.load(f)

    image = Image.open(image_path).convert("RGB")
    words = [item['word'] for item in ocr_result]
    boxes = [item['bounding_box'] for item in ocr_result]

    encoding = processor(
        image,
        words=words,
        boxes=boxes,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    return encoding


In [ ]:
def prepare_test_data(image_path, json_path, processor):
    with open(json_path, 'r') as f:
        ocr_result = json.load(f)

    image = Image.open(image_path).convert("RGB")
    words = [item['word'] for item in ocr_result]
    boxes = [item['bounding_box'] for item in ocr_result]

    encoding = processor(
        image,
        words=words,
        boxes=boxes,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    return encoding


In [ ]:
def prepare_test_data(image_path, json_path, processor):
    with open(json_path, 'r') as f:
        ocr_result = json.load(f)

    image = Image.open(image_path).convert("RGB")
    words = [item['word'] for item in ocr_result]
    boxes = [item['bounding_box'] for item in ocr_result]

    encoding = processor(
        image,
        words=words,
        boxes=boxes,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    return encoding


In [ ]:
def predict(image_path, json_path, model, processor):
    # Prepare the test data
    encoding = prepare_test_data(image_path, json_path, processor)

    # Move tensors to the same device as the model (if using GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Prepare inputs
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    bbox = encoding['bbox'].to(device)
    pixel_values = encoding['pixel_values'].to(device)

    # Perform inference
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values
        )

    # Get predicted label
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return predicted_class


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(test_data_loader, model):
    model.eval()  # Set model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_data_loader:
            # Move batch to the same device as the model (if using GPU)
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            batch = {k: v.to(device) for k, v in batch.items()}

            # Perform inference
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                bbox=batch['bbox'],
                pixel_values=batch['pixel_values']
            )

            # Get predictions and true labels
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            labels = batch['labels'].cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels)

    # Compute metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

# Evaluate the model
evaluate_model(test_data_loader, model)
